# Export checkpoint predictions for ensemble


## Setup


In [ ]:
import os
import sys
import base64
import subprocess
from pathlib import Path

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
repo_dir = Path(os.environ.get("UA_NER_REPO_DIR", root / "UA-Locations-NER-Deep-Learning"))

if not repo_dir.exists():
    if ENVIRONMENT == "colab":
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "-b", "stacked",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env

sys.path.insert(0, str(repo_dir))
print(repo_dir)



In [ ]:
%pip install -q -r "$repo_dir/requirements.txt"


In [ ]:
import gc
from importlib.metadata import version

import pandas as pd
import torch
from google.colab import drive
from tokenizers import Tokenizer

from src.data_io import get_data_dir, load_manifest, load_data_splits, save_json, file_sha256
from src.dataset import prepare_ner_records, prepare_inference_records, make_loader
from src.model import BiLSTMTagger
from src.training import set_seed, predict
from src.export_predictions import export_predictions

drive.mount("/content/drive")


## Checkpoint and tokenizer


In [ ]:
checkpoint_dir = Path("/content/drive/MyDrive/UA-Locations-NER/stacked/YOUR_RUN_FOLDER")
checkpoint_path = checkpoint_dir / "epoch_03.pt"
tokenizer_path = checkpoint_dir / "bpe_bytelevel.json"
splits_dir = repo_dir / "data/splits"
decoding = "viterbi_lines"

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
model_config = checkpoint["model_config"]
train_config = checkpoint["config"]
checkpoint_epoch = checkpoint["epoch"]
checkpoint_provenance = checkpoint.get("provenance", {})
del checkpoint
gc.collect()

checkpoint_sha256 = file_sha256(checkpoint_path)
tokenizer_sha256 = file_sha256(tokenizer_path)
manifest_sha256 = file_sha256(splits_dir / "validation_manifest.parquet")
if tokenizer_sha256 != checkpoint_provenance.get("tokenizer_sha256", tokenizer_sha256):
    raise ValueError("Tokenizer does not match the checkpoint")
if manifest_sha256 != checkpoint_provenance.get("manifest_sha256", manifest_sha256):
    raise ValueError("Validation manifest does not match the checkpoint")

output_dir = Path("/content/drive/MyDrive/ner_results") / (
    f"bilstm_{model_config['num_layers']}l_h{model_config['hidden_size']}"
    f"_s{train_config['seed']}_e{checkpoint_epoch}_{decoding}_{checkpoint_sha256[:8]}"
)

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
set_seed(train_config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMTagger(**model_config).to(device)
tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.no_padding()
tokenizer.no_truncation()
print("Checkpoint:", checkpoint_path)
print("Epoch:", checkpoint_epoch, "decoder:", decoding)
print("Output:", output_dir)


## Full fixed VAL and TEST


In [ ]:
data_dir = get_data_dir(ENVIRONMENT)
manifest = load_manifest(splits_dir)
sources = ("uk", "ru")
expected_val_rows = len(manifest.loc[
    (manifest["split"] == "val") & manifest["source"].isin(sources)
])
del manifest

val_records = []
for source in sources:
    source_splits = load_data_splits(ENVIRONMENT, splits_dir, sources=(source,))
    frame = source_splits[f"{source}_val"][["text", "loc_markers"]]
    del source_splits
    records, skipped = prepare_ner_records(
        tokenizer, frame, source, "val",
        batch_size=train_config.get("prepare_batch_size", 2000),
    )
    val_records.extend(records)
    print(source, "VAL rows:", len(records), "BIO alignment failures retained:", len(skipped))
    del frame, records, skipped
    gc.collect()

if len(val_records) != expected_val_rows:
    raise ValueError("Export requires all fixed validation rows")

test = pd.read_csv(data_dir / "test.csv", usecols=["text_id", "text"])
test_records = prepare_inference_records(tokenizer, test)
loader_options = {
    "pad_id": tokenizer.token_to_id("[PAD]"),
    "batch_size": train_config["batch_size"],
    "seed": train_config["seed"],
    "max_padded_tokens": train_config.get("max_padded_tokens", 16384),
    "pin_memory": device.type == "cuda",
}
val_loader = make_loader(val_records, **loader_options)
test_loader = make_loader(test_records, **loader_options)
print("Full VAL:", len(val_records), "TEST:", len(test_records))


## Export raw spans


In [ ]:
def predict_for_export(model, loader):
    return predict(model, loader, device, decoding=decoding)

export_predictions(
    output_dir=output_dir, checkpoint_path=checkpoint_path,
    model=model, predict_fn=predict_for_export,
    val_records=val_records, val_loader=val_loader,
    test_records=test_records, test_loader=test_loader,
)

save_json(output_dir / "metadata.json", {
    "checkpoint": checkpoint_path.name,
    "checkpoint_path": str(checkpoint_path),
    "checkpoint_sha256": checkpoint_sha256,
    "epoch": checkpoint_epoch,
    "model_config": model_config,
    "training_config": train_config,
    "decoding": decoding,
    "postprocessing": "none",
    "chunking": False,
    "tokenizer": tokenizer_path.name,
    "tokenizer_sha256": tokenizer_sha256,
    "manifest_sha256": manifest_sha256,
    "sources": list(sources),
    "val_rows": len(val_records),
    "test_rows": len(test_records),
    "span_coordinates": "Python Unicode character offsets [start, end) in original text",
    "code_revision": subprocess.check_output(
        ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True,
    ).strip(),
    "source_sha256": {path.name: file_sha256(path)
                      for path in sorted((repo_dir / "src").glob("*.py"))},
    "checkpoint_provenance": checkpoint_provenance,
    "packages": {name: version(name) for name in ["torch", "tokenizers", "numpy", "pandas"]},
})
print(output_dir)
display(pd.DataFrame([
    {"file": path.name, "size_MB": round(path.stat().st_size / 1024**2, 2)}
    for path in sorted(output_dir.iterdir()) if path.is_file()
]))
